# 02 - Data Quality & Imputation

## Purpose

This notebook documents the data-quality assessment, investigation of missing values, and imputation decisions made during data preparation.

It examines missing values, duplicate records, data types, and ZIP-code consistency before establishing appropriate correction and imputation strategies.

## Relationship to the Cleaning Script

The findings and decisions documented here are implemented in the reusable cleaning script:

[`clean_data.py`](../src/data/clean_data.py)

The script applies the established cleaning and imputation procedures to both the training and test datasets and saves the processed files in `data/clean/`:

- [`train_clean.csv`](../data/clean/train_clean.csv)
- [`test_clean.csv`](../data/clean/test_clean.csv)

This notebook documents the analytical reasoning behind the decisions, while the Python script handles their reproducible implementation.

In [2]:
import pandas as pd
from pathlib import Path

In [3]:
ROOT_PATH = Path.cwd().resolve().parents[0]
RAW_DATA_PATH = ROOT_PATH / "data" / "raw"

TRAIN_PATH = RAW_DATA_PATH / "train.csv"

In [4]:
df = pd.read_csv(TRAIN_PATH)

print(f"[*] Loaded Training Dataset shape: {df.shape[0]:,} rows and {df.shape[1]:,} columns.")

[*] Loaded Training Dataset shape: 13,660 rows and 18 columns.


In [5]:
print("="*50)
print("\tMISSING VALUES SUMMARY")
print("="*50)

missing_summary = pd.DataFrame({
    "Missing Values": df.isna().sum(),
    "Missing Percentage": (
        df.isna().mean() * 100
    ).round(2)
})

missing_summary[missing_summary["Missing Values"] > 0]

	MISSING VALUES SUMMARY


,Missing Values,Missing Percentage
City,2,0.01
Metro,106,0.78
RentIndex,1185,8.67


In [5]:
print("="*50)
print("\tDUPLICATE CHECKS")
print("="*50)

print("Exact duplicates:", df.duplicated().sum())
print("Duplicate ZIP-Month pairs:", df.duplicated(["Zip", "Month"]).sum())

	DUPLICATE CHECKS
Exact duplicates: 0
Duplicate ZIP-Month pairs: 0


## ZIP Correction

The ZIP values are saved as integers, these has to be converted into 5 length strings

In [6]:
df['Zip'] = df['Zip'].astype(str).str.zfill(5)
df['Zip'].dtype

<StringDtype(storage='python', na_value=nan)>

## City Correction

Two observations had missing city values, both associated with ZIP code 33194 in Florida. The USPS ZIP Code Lookup confirmed that this ZIP code corresponds to Miami.

Source: [`USPS ZIP Code Lookup`](https://tools.usps.com/zip-code-lookup.htm?citybyzipcode)

In [7]:
df[df['City'].isna()][['City', 'Zip', 'State', 'Metro']]

,City,Zip,State,Metro
1222,NaN,33194,FL,"Miami-Fort Lauderdale-Pompano Beach, FL"
10983,NaN,33194,FL,"Miami-Fort Lauderdale-Pompano Beach, FL"


In [8]:
df.loc[
    df["City"].isna() & (df["Zip"] == "33194"),
    "City"
] = "Miami"

In [9]:
df['City'].isna().sum()

np.int64(0)

## Metro Investigation

In [11]:
metro_missing = df[df["Metro"].isna()]

print("Missing Metro values:", len(metro_missing))
print("Unique ZIP codes affected:", metro_missing["Zip"].nunique())

metro_missing[["Zip", "Month", "State", "City"]].head(10)

Missing Metro values: 106
Unique ZIP codes affected: 53


,Zip,Month,State,City
40,04841,5/1/2026,ME,Rockland
266,05753,5/1/2026,VT,Middlebury
730,04276,5/1/2026,ME,Rumford
732,04769,5/1/2026,ME,Presque Isle
1134,23847,5/1/2026,VA,Emporia
1418,59501,5/1/2026,MT,Havre
2069,22485,5/1/2026,VA,King George
2149,30525,5/1/2026,GA,Clayton
2356,56143,5/1/2026,MN,Jackson
2838,49770,5/1/2026,MI,Petoskey


In [13]:
metro_missing.groupby("Zip")["Month"].nunique().value_counts()

Month
2    53
Name: count, dtype: int64



**Imputation Decision**

All 106 missing values affect 53 ZIP codes, with both monthly observations missing for each ZIP. No corresponding monthly Metro value is available for recovery.

Missing values will therefore be replaced with `"Unknown"` rather than assuming these ZIP codes are outside metropolitan areas.


In [15]:
df["Metro"] = df["Metro"].fillna("Unknown")

print("Remaining missing values:", df["Metro"].isna().sum())
print("Unknown values:", (df["Metro"] == "Unknown").sum())

Remaining missing values: 0
Unknown values: 106


## Rent Index Imputation

In [16]:
rent_missing = df[df["RentIndex"].isna()]

print("Missing values:", len(rent_missing))
print("Affected ZIP codes:", rent_missing["Zip"].nunique())

df.groupby("State")["RentIndex"].agg(
    total="size",
    missing=lambda x: x.isna().sum(),
    median="median"
).assign(
    missing_percent=lambda x: (x["missing"] / x["total"] * 100).round(2)
).sort_values("missing_percent", ascending=False)

Missing values: 1185
Affected ZIP codes: 747


,total,missing,median,missing_percent
State,,,,
VT,24,6,2114.211884,25.00
ME,58,14,2149.866949,24.14
WY,30,7,1357.697240,23.33
AL,224,39,1373.366691,17.41
NH,58,10,2277.876160,17.24
WI,218,36,1370.864582,16.51
AK,32,5,1909.034755,15.62
PA,578,88,1592.049683,15.22
MI,376,57,1488.197240,15.16


In [17]:
rent_by_zip = df.groupby("Zip")["RentIndex"].agg(
    missing_months=lambda x: x.isna().sum(),
    available_months=lambda x: x.notna().sum()
)

rent_by_zip["missing_months"].value_counts().sort_index()

missing_months
0    6083
1     309
2     438
Name: count, dtype: int64

In [18]:
rent_pivot = df.pivot(
    index="Zip",
    columns="Month",
    values="RentIndex"
)

complete_rent = rent_pivot.sort_index(axis=1).dropna()

monthly_change = (
    (complete_rent["6/1/2026"] - complete_rent["5/1/2026"])
    / complete_rent["5/1/2026"]
) * 100

monthly_change.describe()

count    6083.000000
mean        0.478490
std         1.534130
min       -13.178745
25%        -0.280221
50%         0.442955
75%         1.173226
max        15.752106
dtype: float64



**Imputation Strategy**

A two-stage imputation approach is used:

1. For ZIP codes with one missing monthly value, use the available `RentIndex` value from the same ZIP code's other month.
2. For ZIP codes missing both monthly values, use the median `RentIndex` for the corresponding state.

State medians are calculated from the original observed values before imputation.



In [20]:
state_rent_medians = df.groupby("State")["RentIndex"].median()

zip_rent_medians = df.groupby("Zip")["RentIndex"].transform("median")
df["RentIndex"] = df["RentIndex"].fillna(zip_rent_medians)

df["RentIndex"] = df["RentIndex"].fillna(
    df["State"].map(state_rent_medians)
)

In [21]:
print("Remaining missing RentIndex values:", df["RentIndex"].isna().sum())

df["RentIndex"].describe()

Remaining missing RentIndex values: 0


count    13660.000000
mean      2052.965499
std       1786.872764
min        722.931319
25%       1471.245986
50%       1805.886685
75%       2366.512590
max      97151.633120
Name: RentIndex, dtype: float64



**Result**

All 1,185 missing `RentIndex` values were addressed using the two-stage imputation strategy. The original observed values were retained, while missing values were estimated using available monthly or state-level information.



## Final Validation

Verify that the cleaning and imputation steps have been applied successfully and that the dataset retains its expected structure.



In [22]:
print("Dataset shape:", df.shape)
print("Remaining missing values:", df.isna().sum().sum())
print("Exact duplicates:", df.duplicated().sum())
print("Duplicate ZIP-Month pairs:", df.duplicated(["Zip", "Month"]).sum())
print("Unique ZIP codes:", df["Zip"].nunique())
print("ZIP lengths:", df["Zip"].str.len().value_counts().to_dict())
print(
    "Target range:",
    df["AffordabilityPercentageTrue"].min(),
    "to",
    df["AffordabilityPercentageTrue"].max()
)

assert df.isna().sum().sum() == 0
assert df.duplicated().sum() == 0
assert df.duplicated(["Zip", "Month"]).sum() == 0
assert df["Zip"].str.len().eq(5).all()

Dataset shape: (13660, 18)
Remaining missing values: 0
Exact duplicates: 0
Duplicate ZIP-Month pairs: 0
Unique ZIP codes: 6830
ZIP lengths: {5: 13660}
Target range: 0 to 100
